In [40]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [41]:
torch.manual_seed(42)

In [42]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [43]:
df = pd.read_csv('fashion-mnist_train.csv')
df.sample(5)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
31185,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7337,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59639,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21991,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
48826,0,0,0,0,0,0,0,0,0,0,...,68,31,21,0,0,0,0,0,0,0


In [44]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values


In [45]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [46]:
X_train = X_train/255.0
X_test = X_test/255.0

In [47]:
from numpy import dtype
class CustomDataset(Dataset):

    def __init__(self, features, labels):

        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):

        return len(self.features)

    def __getitem__(self, index):

        return self.features[index], self.labels[index]

In [48]:
train_dataset = CustomDataset(X_train, y_train)

In [49]:
test_dataset = CustomDataset(X_test, y_test)

In [51]:
%pip install optuna

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
class MyNN(nn.Module):

    def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):
        super().__init__()
        layers = []
        for i in range(num_hidden_layers):
            layers.append(nn.Linear(input_dim, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            input_dim = neurons_per_layer
        
        layers.append(nn.Linear(neurons_per_layer, output_dim))  
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        
        return self.model(x)

In [56]:
def objective(trial):
    
    num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 5)
    neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step=8)
    epochs = trial.suggest_int("epochs", 10, 50, step=10)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
    optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    
    input_dim = 784
    output_dim = 10
    
    model = MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    
    if optimizer_name=='Adam':
        optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name=='SGD':
        optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    else:
        optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    for epoch in range(epochs):

        for batch_features, batch_labels in train_loader:

            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

            outputs = model(batch_features)

            loss = criterion(outputs, batch_labels)

            optimizer.zero_grad()
            loss.backward()

            optimizer.step()

            
    model.eval()    
    
    total = 0
    correct = 0

    with torch.no_grad():

        for batch_features, batch_labels in test_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)
            _, predicted = torch.max(outputs, 1)

            total += batch_labels.shape[0]
            correct += (predicted == batch_labels).sum().item()
        
        accuracy = correct/total


    return accuracy
    

In [57]:
import optuna

study = optuna.create_study(direction='maximize')


[I 2026-08-26 13:25:36,289] A new study created in memory with name: no-name-1b8e70b4-71db-4394-aee1-639e43d3c1f1


In [58]:
study.optimize(objective, n_trials=10)


[I 2026-08-26 13:26:55,093] Trial 0 finished with value: 0.43183333333333335 and parameters: {'num_hidden_layers': 4, 'neurons_per_layer': 32, 'epochs': 40, 'learning_rate': 3.1776106629908986e-05, 'dropout_rate': 0.30000000000000004, 'batch_size': 128, 'optimizer': 'SGD', 'weight_decay': 0.00024465110923748605}. Best is trial 0 with value: 0.43183333333333335.
[I 2026-08-26 13:29:57,943] Trial 1 finished with value: 0.8815833333333334 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 80, 'epochs': 50, 'learning_rate': 0.0038363535585983705, 'dropout_rate': 0.2, 'batch_size': 64, 'optimizer': 'SGD', 'weight_decay': 0.0009473632709587348}. Best is trial 1 with value: 0.8815833333333334.
[I 2026-08-26 13:33:28,866] Trial 2 finished with value: 0.84775 and parameters: {'num_hidden_layers': 4, 'neurons_per_layer': 96, 'epochs': 20, 'learning_rate': 0.0004107283285470507, 'dropout_rate': 0.30000000000000004, 'batch_size': 16, 'optimizer': 'RMSprop', 'weight_decay': 4.96147853816

In [59]:
study.best_value

0.8815833333333334

In [60]:
study.best_params

{'num_hidden_layers': 5,
 'neurons_per_layer': 80,
 'epochs': 50,
 'learning_rate': 0.0038363535585983705,
 'dropout_rate': 0.2,
 'batch_size': 64,
 'optimizer': 'SGD',
 'weight_decay': 0.0009473632709587348}